
# NB-C — Pipeline (PipelineDecorator) — ClearML x ResNet (CIFAR-10)

**Steps**:
1. `prepare_data`: télécharge le dataset de feedback ClearML (dernière version publiée)
2. `train`: entraîne un ResNet18 (torchvision) sur CIFAR-10 + feedback
3. `register_model`: compare avec la prod actuelle et met à jour les tags `staging` / `production`

> Exécution: la pipeline se lance via **queue `default`** (prise par l'agent du NB-D).


In [1]:
!pip -q install clearml



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
%env CLEARML_WEB_HOST=https://app.clear.ml/
%env CLEARML_API_HOST=https://api.clear.ml
%env CLEARML_FILES_HOST=https://files.clear.ml
# tp3
%env CLEARML_API_ACCESS_KEY=VES7TTVL4QDF36XPHCPXLAV1KDIZY4
%env CLEARML_API_SECRET_KEY=ygijnEsVvuMODXfwaXay-EIcKZbQVReuv18Bd9O5qskRWuHsAbn_V0w1b5waRQbgaNE

env: CLEARML_WEB_HOST=https://app.clear.ml/
env: CLEARML_API_HOST=https://api.clear.ml
env: CLEARML_FILES_HOST=https://files.clear.ml
env: CLEARML_API_ACCESS_KEY=VES7TTVL4QDF36XPHCPXLAV1KDIZY4
env: CLEARML_API_SECRET_KEY=ygijnEsVvuMODXfwaXay-EIcKZbQVReuv18Bd9O5qskRWuHsAbn_V0w1b5waRQbgaNE


In [3]:
%pip install -q torchvision tqdm clearml
import os
import urllib.request
from tqdm import tqdm
from torchvision import datasets
from clearml import Dataset

home_dir = os.path.expanduser("~")
data_root = os.path.join(home_dir, "cifar10_data")
archive_path = os.path.join(data_root, "cifar-10-python.tar.gz")
os.makedirs(data_root, exist_ok=True)

url = "https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz"

# Classe pour afficher la barre de progression tqdm pendant le téléchargement
class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

# Téléchargement robuste avec barre de progression
if not os.path.exists(archive_path) or os.path.getsize(archive_path) < 100_000_000:
    print("Téléchargement de CIFAR-10 avec suivi de progression...")
    try:
        with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=url.split('/')[-1]) as t:
            urllib.request.urlretrieve(url, filename=archive_path, reporthook=t.update_to)
        print("Téléchargement complété avec succès !")
    except Exception as e:
        print(f"Le miroir principal a échoué ({e}). Utilisation du miroir secondaire...")
        backup_url = "https://fast-mirror-cifar.s3.amazonaws.com/cifar-10-python.tar.gz"
        with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=backup_url.split('/')[-1]) as t:
            urllib.request.urlretrieve(backup_url, filename=archive_path, reporthook=t.update_to)

# Extraction locale via torchvision
print("Extraction locale...")
train_ds = datasets.CIFAR10(root=data_root, train=True, download=True)
test_ds = datasets.CIFAR10(root=data_root, train=False, download=True)

print(f"Train: {len(train_ds)} | Test: {len(test_ds)}")

# Envoi vers ClearML
dataset_dir = os.path.join(data_root, "cifar-10-batches-py")
new_dataset = Dataset.create(
    dataset_project="Image-Classifier/Gradio-Retrain-Demo-IABD",
    dataset_name="cifar10-base"
)
new_dataset.add_files(path=dataset_dir)
new_dataset.upload()
new_dataset.finalize()

print("CIFAR-10 importé avec succès sur ClearML !")

error: externally-managed-environment

× This environment is externally managed
╰─> This Python installation is managed by uv and should not be modified.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.
Note: you may need to restart the kernel to use updated packages.
Extraction locale...
Files already downloaded and verified
Files already downloaded and verified
Train: 50000 | Test: 10000
ClearML results page: https://app.clear.ml/projects/b28a1059b808401aae2db1db387cfbca/tasks/e9baf381f7b240a684edbba87a2d2d60/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/b28a1059b808401aae2db1db387cfbca/experiments/e9baf381f7b240a684edbba87a2d2d60
Generating SHA2 hash for 8 files


100%|██████████| 8/8 [00:01<00:00,  5.91it/s]


Hash generation completed
Uploading dataset changes (8 files compressed to 172.46 MiB) to https://files.clear.ml


█████████████████████████████ 100% | 172.46/172.46 MB [01:36<00:00,  1.80MB/s]: 


File compression and upload completed: total size 172.46 MiB, 1 chunk(s) stored (average size 172.46 MiB)
CIFAR-10 importé avec succès sur ClearML !


In [ ]:
# NB-C — ClearML Pipeline (ResNet18 CIFAR-10)
from pathlib import Path
import os

from clearml import Task
from clearml.automation.controller import PipelineDecorator

PROJECT = "Image-Classifier/Gradio-Retrain-Demo-IABD"

# Use a writable local directory instead of /content, which is only valid in Colab.
OUTPUT_ROOT = Path.home() / "clearml_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

task = Task.init(project_name=PROJECT, task_name="retrain_pipeline_controller", task_type=Task.TaskTypes.controller)
print("Controller Task ID:", task.id)

# CPU-only setup: keep the worker aligned with the local CPU environment.
task.set_packages({
    "clearml": "==2.0.2",
    "torch": "==2.14.1",
    "torchvision": "==0.29.1",
})

@PipelineDecorator.component(
    name="step_prepare_data",
    execution_queue="default",
    return_values=["feedback_dir"]
)
def step_prepare_data():
    from clearml import Dataset
    PROJECT = "Image-Classifier/Gradio-Retrain-Demo-IABD"
    try:
        # On force la vérification du dataset et on contourne le cache corrompu avec force_download=True
        ds = Dataset.get(dataset_name="image-feedback", dataset_project=PROJECT)
        return ds.get_local_copy(force_download=True)
    except Exception as e:
        print("No feedback dataset → train on CIFAR-10 only.", e)
        return ""

@PipelineDecorator.component(
    name="step_train",
    execution_queue="default",
    return_values=["model_id","metrics_dict"]
)
def step_train(feedback_dir: str):
    import os, time, shutil
    from pathlib import Path
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    from torch.utils.data import DataLoader, ConcatDataset
    from torchvision import datasets, transforms, models
    from clearml import Task as _Task, Dataset as _Dataset

    # Force CPU execution to match the local environment; this avoids GPU-specific package mismatches.
    DEVICE = "cpu"
    BATCH_SIZE, EPOCHS, LR = 128, 3, 1e-3

    TRANSFORM_TRAIN = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
    TRANSFORM_TEST = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

    def build_resnet18(num_classes=10):
        m = models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m

    @torch.no_grad()
    def eval_acc(model, loader):
        model.eval(); correct=total=0
        for x,y in loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(1)
            correct += int((pred==y).sum().item()); total += int(y.numel())
        return correct/total

    def train_one_model(train_ds, test_ds):
        model = build_resnet18(10).to(DEVICE)
        tl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
        vl = DataLoader(test_ds,  batch_size=256,     shuffle=False, num_workers=2)
        opt = optim.Adam(model.parameters(), lr=LR); best=0.0
        for ep in range(EPOCHS):
            model.train()
            for x,y in tl:
                x,y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad(); loss = F.cross_entropy(model(x), y); loss.backward(); opt.step()
            acc = eval_acc(model, vl)
            _Task.current_task().get_logger().report_scalar("valid","acc",acc, iteration=ep)
            best = max(best, acc); print(f"[{ep+1}/{EPOCHS}] acc={acc:.4f} best={best:.4f}")
        return model, best

    # 📦 RÉCUPÉRATION DU DATASET CACHÉ SUR CLEARML
    print("Récupération locale du dataset CIFAR-10 depuis ClearML...")
    cifar_ds = _Dataset.get(dataset_name="cifar10-base", dataset_project="Image-Classifier/Gradio-Retrain-Demo-IABD")
    cifar_local_path = cifar_ds.get_local_copy()

    # CIFAR10 cherche un sous-dossier 'cifar-10-batches-py' sous root.
    # Si le dataset ClearML contient directement les fichiers, on crée le sous-dossier pour torchvision.
    target_dir = os.path.join(cifar_local_path, "cifar-10-batches-py")
    if not os.path.exists(target_dir):
        os.makedirs(target_dir, exist_ok=True)
        for item in os.listdir(cifar_local_path):
            item_path = os.path.join(cifar_local_path, item)
            if os.path.isfile(item_path):
                shutil.copy(item_path, target_dir)

    train_base = datasets.CIFAR10(root=cifar_local_path, train=True,  download=False, transform=TRANSFORM_TRAIN)
    test_base  = datasets.CIFAR10(root=cifar_local_path, train=False, download=False, transform=TRANSFORM_TEST)

    if feedback_dir and os.path.isdir(feedback_dir):
        try:
            fb = datasets.ImageFolder(root=feedback_dir, transform=TRANSFORM_TRAIN)
            train_ds = ConcatDataset([train_base, fb]); print(f"Training on CIFAR-10 + feedback ({len(fb)} images)")
        except Exception as e:
            print("Feedback load failed → base only.", e); train_ds = train_base
    else:
        train_ds = train_base

    model, best = train_one_model(train_ds, test_base)
    out = OUTPUT_ROOT
    out.mkdir(parents=True, exist_ok=True)
    mp = str(out / f"resnet18_{int(time.time())}.pt")
    torch.save(model.state_dict(), mp)

    t = _Task.current_task()
    ts = int(time.time())
    t.update_output_model(
        model_path=mp,
        name=f"resnet18-{ts}",
        comment=f"candidate (valid_acc={best:.4f})",
        tags=["candidate","resnet18"]
    )
    mid = t.models["output"][-1].id
    return mid, {"valid_acc": float(best)}

@PipelineDecorator.component(
    name="step_register_model",
    execution_queue="default",
    return_values=["improved","new_model_id"]
)
def step_register_model(model_id: str, metrics_dict: dict):
    from clearml import Model
    PROJECT = "Image-Classifier/Gradio-Retrain-Demo-IABD"
    new_acc = float(metrics_dict.get("valid_acc", 0.0))
    try:
        prods = Model.query_models(project_name=PROJECT, tags=["production"], only_published=True) or []
        try:
            prod_acc = float(prods[-1].get_metadata_casted("valid_acc"))
        except Exception:
            prod_acc = 0.0
    except Exception:
        prods, prod_acc = [], 0.0
    improved = new_acc >= prod_acc
    m = Model(model_id=model_id)
    m.publish()
    try:
        m.set_metadata("valid_acc", str(new_acc))
    except Exception:
        pass
    if improved:
        try:
            if prods: prods[-1].archive()
        except Exception:
            pass
        try:
            m.set_tags(list(set((m.tags or []) + ["production"])))
        except Exception:
            pass
    else:
        try:
            m.set_tags(list(set((m.tags or []) + ["staging"])))
        except Exception:
            pass
    print(f"Registered model: {model_id} | improved={improved} | new_acc={new_acc:.4f} vs prod_acc={prod_acc:.4f}")
    return bool(improved), model_id

@PipelineDecorator.pipeline(
    name="retrain_pipeline",
    project=PROJECT,
    version="1.0",
    pipeline_execution_queue=None
)
def pipeline():
    fb = step_prepare_data()
    mid, metrics = step_train(fb)
    step_register_model(mid, metrics)


ClearML Task: overwriting (reusing) task id=33714fe795ea49a8adffdfb55770e1b1
ClearML results page: https://app.clear.ml/projects/dc7beb331d174ba3a0cfc2eed86fac20/tasks/33714fe795ea49a8adffdfb55770e1b1/output/log


Could not fetch GPU stats: NVML Shared Library Not Found


Controller Task ID: 33714fe795ea49a8adffdfb55770e1b1


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


In [5]:

# Déclenchement (non bloquant côté notebook, le contrôleur part sur l'agent)
pipeline()


Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module


ClearML pipeline page: https://app.clear.ml/pipelines/dc7beb331d174ba3a0cfc2eed86fac20/experiments/33714fe795ea49a8adffdfb55770e1b1


Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module
Could not fetch function declared in __main__: <module '__main__'> is a built-in module
Could not fetch function imports: <module '__main__'> is a built-in module


Launching step [step_prepare_data]
Launching step [step_train]


ValueError: Pipeline step "step_train", Task ID=b7edc654001a4fc6ba9105252882efca failed